In [1]:
!pip install evidently

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.0/238.0 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 579.2/579.2 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.4 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
from evidently import Report
from evidently.presets import DataDriftPreset
import pickle
import json

In [6]:
from google.colab import files
uploaded = files.upload()  # CSV file select karo

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print('Shape:', df.shape)

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv
Shape: (7043, 21)


In [7]:
# Reference data — purana data (pehle 5000 rows)
reference_data = df.iloc[:5000].copy()

# Current data — naya data (baaki rows)
current_data = df.iloc[5000:].copy()

print(f'Reference data: {reference_data.shape}')
print(f'Current data: {current_data.shape}')

Reference data: (5000, 21)
Current data: (2043, 21)


In [10]:
import evidently
print(evidently.__version__)

0.7.21


In [12]:
print([m for m in dir(report) if not m.startswith('_')])

['include_tests', 'items', 'metadata', 'metrics', 'run', 'set_batch_size', 'set_dataset_id', 'set_model_id', 'set_reference_id', 'tags']


In [14]:
print('=== Data Drift Detection Report ===\n')

drift_results = []

for col in reference_data.select_dtypes(include=[np.number]).columns:
    ref_mean = reference_data[col].mean()
    cur_mean = current_data[col].mean()
    drift_pct = abs((cur_mean - ref_mean) / (ref_mean + 1e-10)) * 100

    status = 'DRIFT DETECTED ⚠️' if drift_pct > 10 else 'No Drift ✅'
    drift_results.append({'Feature': col, 'Ref Mean': round(ref_mean, 2),
                          'Current Mean': round(cur_mean, 2),
                          'Drift %': round(drift_pct, 2), 'Status': status})

drift_df = pd.DataFrame(drift_results)
print(drift_df.to_string(index=False))

=== Data Drift Detection Report ===

       Feature  Ref Mean  Current Mean  Drift %     Status
 SeniorCitizen      0.16          0.17     5.50 No Drift ✅
        tenure     32.26         32.65     1.21 No Drift ✅
MonthlyCharges     64.78         64.71     0.11 No Drift ✅


In [15]:
drift_df.to_csv('monitoring_report.csv', index=False)
print('Monitoring report saved!')

from google.colab import files
files.download('monitoring_report.csv')

Monitoring report saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>